# 04 — Napredne Neuronske Mreže
**Cilj:** Istražiti napredne NN tehnike i arhitekture za poboljšanje baseline MLP-a:
- Batch Normalization
- Residual / Skip Connections
- Focal Loss (specijalno dizajniran za imbalanced probleme)
- Learning Rate Scheduling
- Hyperparameter Tuning (Keras Tuner)

---

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import keras_tuner as kt

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

PROC    = '../data/processed/'
MODELS  = '../models/'
RESULTS = '../results/'

print(f'TensorFlow: {tf.__version__}')

## 1. Učitavanje podataka

In [ ]:
X_train       = np.load(f'{PROC}X_train.npy')
X_val         = np.load(f'{PROC}X_val.npy')
X_test        = np.load(f'{PROC}X_test.npy')
y_train       = np.load(f'{PROC}y_train.npy')
y_val         = np.load(f'{PROC}y_val.npy')
y_test        = np.load(f'{PROC}y_test.npy')
X_train_smote = np.load(f'{PROC}X_train_smote.npy')
y_train_smote = np.load(f'{PROC}y_train_smote.npy')

with open(f'{PROC}class_weights.json') as f:
    cw = json.load(f)
class_weight = {int(k): v for k, v in cw.items()}

INPUT_DIM = X_train.shape[1]
print(f'Input dim: {INPUT_DIM}')

## 2. Focal Loss

Focal Loss je dizajniran za ekstremno imbalanced probleme. Smanjuje doprinos lako klasifikovanih uzoraka (legitimne tx) i fokusira se na teške slučajeve (fraud).

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- **γ (gamma)**: modulating factor — tipično 2.0
- **α (alpha)**: balansira pozitivne/negativne uzorke — tipično ~0.25

In [ ]:
class FocalLoss(keras.losses.Loss):
    """Focal Loss za imbalanced binary classification."""
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        bce    = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t    = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        fl     = alpha_t * tf.pow(1 - p_t, self.gamma) * bce
        return tf.reduce_mean(fl)

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha})
        return config

print('✅ FocalLoss definisan')

## 3. Model A — Deep MLP sa Batch Normalization

In [ ]:
def build_bn_mlp(input_dim, hidden_units=[128, 64, 32, 16], dropout_rate=0.3, lr=1e-3):
    """
    MLP sa Batch Normalization između slojeva.
    BN stabilizuje trening, ubrzava konvergenciju i može zameniti Dropout (ali kombinujemo oba).
    Redosled: Dense -> BN -> Activation -> Dropout (preporučeni redosled)
    """
    model = keras.Sequential(name='Deep_BN_MLP')
    model.add(layers.Input(shape=(input_dim,)))
    
    for units in hidden_units:
        model.add(layers.Dense(units, kernel_initializer='he_normal',
                               kernel_regularizer=keras.regularizers.l2(1e-4)))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=FocalLoss(gamma=2.0, alpha=0.25),
        metrics=[keras.metrics.AUC(curve='PR', name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

model_bn = build_bn_mlp(INPUT_DIM)
model_bn.summary()

In [ ]:
cb_bn = [
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=12,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max',
                                 factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(f'{MODELS}mlp_bn.keras', monitor='val_auc',
                               mode='max', save_best_only=True)
]

history_bn = model_bn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120, batch_size=256,
    class_weight=class_weight,
    callbacks=cb_bn, verbose=1
)

## 4. Model B — Residual Network (Skip Connections)

In [ ]:
def residual_block(x, units, dropout_rate=0.3):
    """Residual blok: main path + skip connection."""
    # Main path
    h = layers.Dense(units, kernel_initializer='he_normal',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    h = layers.BatchNormalization()(h)
    h = layers.Activation('relu')(h)
    h = layers.Dropout(dropout_rate)(h)
    h = layers.Dense(units, kernel_initializer='he_normal',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(h)
    h = layers.BatchNormalization()(h)
    
    # Skip connection: projekcija ako dimenzija nije ista
    if x.shape[-1] != units:
        x = layers.Dense(units, use_bias=False)(x)
    
    out = layers.Add()([h, x])        # Skip connection
    out = layers.Activation('relu')(out)
    out = layers.Dropout(dropout_rate)(out)
    return out


def build_resnet_mlp(input_dim, units=[64, 64, 32, 32], dropout_rate=0.3, lr=1e-3):
    """
    Residual MLP (ResNet-inspired) za tabelarne podatke.
    Skip connections pomažu gradientu da prođe dublje slojeve (vanishing gradient problem).
    """
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(units[0], activation='relu',
                     kernel_initializer='he_normal')(inp)  # Ulazna projekcija
    
    # Parovi slojeva sa residual vezama
    for i in range(0, len(units) - 1, 2):
        x = residual_block(x, units[i], dropout_rate)
    
    out = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs=inp, outputs=out, name='ResNet_MLP')
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=FocalLoss(gamma=2.0, alpha=0.25),
        metrics=[keras.metrics.AUC(curve='PR', name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

model_res = build_resnet_mlp(INPUT_DIM)
model_res.summary()

In [ ]:
cb_res = [
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=12,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max',
                                 factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(f'{MODELS}mlp_resnet.keras', monitor='val_auc',
                               mode='max', save_best_only=True)
]

history_res = model_res.fit(
    X_train_smote, y_train_smote,
    validation_data=(X_val, y_val),
    epochs=120, batch_size=256,
    callbacks=cb_res, verbose=1
)

## 5. Model C — Keras Tuner (Automatski Hyperparameter Search)

In [ ]:
def build_tunable_model(hp):
    """Model čiji se hiperparametri pretražuju automatski."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(INPUT_DIM,)))
    
    n_layers = hp.Int('n_layers', 2, 5)
    for i in range(n_layers):
        units = hp.Int(f'units_{i}', min_value=16, max_value=256, step=16)
        model.add(layers.Dense(units, kernel_initializer='he_normal',
                               kernel_regularizer=keras.regularizers.l2(
                                   hp.Float('l2', 1e-5, 1e-3, sampling='log'))))
        
        if hp.Boolean(f'use_bn_{i}'):
            model.add(layers.BatchNormalization())
        
        model.add(layers.Activation(
            hp.Choice(f'activation_{i}', ['relu', 'elu', 'selu'])))
        model.add(layers.Dropout(
            hp.Float(f'dropout_{i}', 0.1, 0.5, step=0.1)))
    
    model.add(layers.Dense(1, activation='sigmoid'))
    
    lr = hp.Float('lr', 1e-4, 1e-2, sampling='log')
    gamma = hp.Float('focal_gamma', 1.0, 3.0, step=0.5)
    
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=FocalLoss(gamma=gamma, alpha=0.25),
        metrics=[keras.metrics.AUC(curve='PR', name='auc')]
    )
    return model


tuner = kt.BayesianOptimization(
    build_tunable_model,
    objective=kt.Objective('val_auc', direction='max'),
    max_trials=20,
    seed=RANDOM_SEED,
    directory='../models/tuner',
    project_name='fraud_nn'
)

tuner.search_space_summary()

In [ ]:
cb_tuner = [
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, verbose=0)
]

print('Pokrećem Bayesian Optimization pretragu (20 trials)...')
tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=cb_tuner,
    verbose=0
)

print('\n=== BEST HYPERPARAMETERS ===')
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print(best_hp.values)

In [ ]:
# Treniraj best model do kraja
model_tuned = tuner.hypermodel.build(best_hp)

cb_tuned = [
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15,
                             restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(f'{MODELS}mlp_tuned.keras', monitor='val_auc',
                               mode='max', save_best_only=True)
]

history_tuned = model_tuned.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150, batch_size=256,
    class_weight=class_weight,
    callbacks=cb_tuned, verbose=1
)

## 6. Helper: evaluacija i poređenje

In [ ]:
def find_best_threshold(model, X_val, y_val):
    y_prob = model.predict(X_val, verbose=0).ravel()
    thresholds = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y_val, (y_prob >= t).astype(int)) for t in thresholds]
    best_t = thresholds[np.argmax(f1s)]
    print(f'Optimalni threshold: {best_t:.2f} → F1 = {max(f1s):.4f}')
    return best_t


def quick_eval(model, X_test, y_test, title, val_X=None, val_y=None):
    t = find_best_threshold(model, val_X or X_test, val_y or y_test)
    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= t).astype(int)
    return {
        'title': title,
        'roc_auc': roc_auc_score(y_test, y_prob),
        'pr_auc':  average_precision_score(y_test, y_prob),
        'f1':      f1_score(y_test, y_pred),
        'threshold': t
    }


results = [
    quick_eval(model_bn,    X_test, y_test, 'Deep BN MLP + Focal', X_val, y_val),
    quick_eval(model_res,   X_test, y_test, 'ResNet MLP + Focal',  X_val, y_val),
    quick_eval(model_tuned, X_test, y_test, 'Tuned MLP (BayesOpt)',X_val, y_val),
]

results_df = pd.DataFrame(results).set_index('title')

print('\n=== POREĐENJE NAPREDNIH MODELA ===')
display(results_df.style.highlight_max(color='#90EE90', subset=['roc_auc', 'pr_auc', 'f1']))

In [ ]:
# Vizualizacija
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, color in zip(axes, ['roc_auc', 'pr_auc', 'f1'], ['steelblue', 'coral', 'seagreen']):
    bars = ax.barh(results_df.index, results_df[metric], color=color, edgecolor='black', alpha=0.85)
    ax.set_title(metric.upper(), fontweight='bold')
    ax.set_xlim(0.7, 1.0)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Napredni NN Modeli — Poređenje Metrika', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}04_advanced_comparison.png', bbox_inches='tight')
plt.show()

results_df.to_csv(f'{RESULTS}04_advanced_results.csv')
print('\n→ Sledeći notebook: 05_autoencoder.ipynb')